# NumCompute Quickstart

A short tour of each NumCompute module, with runnable examples.

**To run it:**
1. From the project root, run `uv sync` (see the README).
2. Open this notebook and choose the `.venv` Python as the kernel. If VS Code asks to install `ipykernel`, accept.
3. Run all cells from top to bottom. Paths assume the notebook runs from the `demo/` folder.

In [1]:
from pathlib import Path

import numpy as np

DATA_DIR = Path("..") / "tests" / "test_csv_files"

## 1. Loading data (`io`)

`load_csv` reads a CSV into a 2D float array. Missing values become `NaN`.

In [2]:
from numcompute.io import load_csv

X = load_csv(DATA_DIR / "missing_data_multiple.csv")
print(X.shape)
X

(7, 3)


array([[ 1.,  2., nan],
       [ 4.,  5.,  6.],
       [ 7., nan,  9.],
       [10., 11., 12.],
       [13., 14., 15.],
       [16., 17., nan],
       [19., 20., 21.]])

## 2. Preprocessing (`preprocessing`)

Every transformer has `fit(X)`, `transform(X)` and `fit_transform(X)`. Calling `transform` before `fit` raises `NotFittedError`.

### Filling missing values: `SimpleImputer`

In [3]:
from numcompute.preprocessing import SimpleImputer

X_filled = SimpleImputer(fill_value=0.0).fit_transform(X)
X_filled

array([[ 1.,  2.,  0.],
       [ 4.,  5.,  6.],
       [ 7.,  0.,  9.],
       [10., 11., 12.],
       [13., 14., 15.],
       [16., 17.,  0.],
       [19., 20., 21.]])

### Scaling: `StandardScaler` and `MinMaxScaler`

`StandardScaler` gives each column mean 0 and standard deviation 1. `MinMaxScaler` maps each column into `feature_range`.

In [4]:
from numcompute.preprocessing import MinMaxScaler, StandardScaler

X_std = StandardScaler().fit_transform(X_filled)
print("column means:", X_std.mean(axis=0).round(6) + 0.0)
print("column stds: ", X_std.std(axis=0).round(6))

X_minmax = MinMaxScaler(feature_range=(-1, 1)).fit_transform(X_filled)
X_minmax

column means: [0. 0. 0.]
column stds:  [1. 1. 1.]


array([[-1.        , -0.8       , -1.        ],
       [-0.66666667, -0.5       , -0.42857143],
       [-0.33333333, -1.        , -0.14285714],
       [ 0.        ,  0.1       ,  0.14285714],
       [ 0.33333333,  0.4       ,  0.42857143],
       [ 0.66666667,  0.7       , -1.        ],
       [ 1.        ,  1.        ,  1.        ]])

### Learn on training data, apply to new data

`fit` learns from one dataset; `transform` reuses what was learned on any new data with the same columns.

In [5]:
scaler = StandardScaler().fit(X_filled)

new_data = np.array([[5.0, 5.0, 5.0]])
scaler.transform(new_data)

array([[-0.83333333, -0.6821865 , -0.55777335]])

### Categorical data: `OneHotEncoder`

Each column becomes one 0/1 column per category (sorted). Unknown categories raise an error unless `handle_unknown="ignore"`.

In [6]:
from numcompute.preprocessing import OneHotEncoder

colours = [["red", "S"], ["blue", "L"], ["red", "L"]]

encoder = OneHotEncoder().fit(colours)
print(encoder.categories_)
encoder.transform(colours)

[array(['blue', 'red'], dtype='<U4'), array(['L', 'S'], dtype='<U4')]


array([[0., 1., 0., 1.],
       [1., 0., 1., 0.],
       [0., 1., 1., 0.]])

In [7]:
OneHotEncoder(handle_unknown="ignore").fit(colours).transform([["green", "S"]])

array([[0., 0., 0., 1.]])

## 3. Numerical gradients (`optim`)

`grad` and `jacobian` estimate derivatives with finite differences. `method="central"` (default) is more accurate than `"forward"`.

The gradient of $f(x) = \sum x_i^2$ is $2x$:

In [8]:
from numcompute.optim import grad, jacobian


def sum_of_squares(x):
    return np.sum(x**2)


x = np.array([1.0, 2.0, 3.0])
print("central:", grad(sum_of_squares, x))
print("forward:", grad(sum_of_squares, x, method="forward"))

central: [2. 4. 6.]
forward: [2.00001 4.00001 6.00001]


For a function with several outputs, `jacobian` returns shape `(outputs, inputs)`. For $F(x) = [x_0 x_1,\ \sin x_0]$ the exact answer is $\begin{bmatrix} x_1 & x_0 \\ \cos x_0 & 0 \end{bmatrix}$:

In [9]:
def vector_func(x):
    return np.array([x[0] * x[1], np.sin(x[0])])


jacobian(vector_func, [1.0, 2.0])

array([[2.        , 1.        ],
       [0.54030231, 0.        ]])

## 4. Sort and Search (`sort_search`)

`sort_search` provides stable and multi-key sorting, efficient top-k selection, manual Quickselect, and binary search.

In [10]:
from numcompute.sort_search import (
    stable_sort,
    multi_key_sort,
    topk,
    quickselect,
    binary_search,
)

In [11]:
### Stable sorting

values = np.array([5, 2, 4, 2, 1])

stable_sort(values)

array([1, 2, 2, 4, 5])

### Multi-key sorting

`multi_key_sort` sorts the rows of a 2D array by using its column indices in order of priority. The first key is always the sorting key.

In [12]:
data = np.array([
    [2, 30],
    [1, 20],
    [2, 10],
    [1, 10],
])

multi_key_sort(data, [0,1])

array([[ 1, 10],
       [ 1, 20],
       [ 2, 10],
       [ 2, 30]])

### Top-k selection

`topk` uses partial sorting to efficiently return the largest of the smallest `k` values

In [13]:
values = np.array([4, 11, 2, 19, 7])

print("largest 3 values:", topk(values, 3, largest=True, return_indices=False))
print("largest 3 indices:", topk(values, 3, largest=True, return_indices=True))
print("smallest 2 values:", topk(values, 2, largest=False, return_indices=False))

largest 3 values: [19 11  7]
largest 3 indices: [3 1 4]
smallest 2 values: [2 4]


### Quickselect

`quickselect` finds the k-th smallest value without fully sorting the array. `k` uses 1-based indexing. The algorithm modifies its input array in place, so copy is used below.

In [14]:
values = np.array([6, 1, 8, 0, 5, 3])

third_smallest = quickselect(values.copy(), 3)

print("3rd smallest is:", third_smallest)

3rd smallest is: 3


### Binary search

`binary_search` searches a sorted array and returns the index and a boolean indicating whether the value was present in the array. If the value was absent, i.e. the Boolean was False, then that index is where it is inserted.

In [15]:
sorted_values = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])

print("Search for 5:", binary_search(sorted_values, 5))
print("Search for 10:", binary_search(sorted_values, 10))

Search for 5: (4, True)
Search for 10: (9, False)


## 8. Pipelines (`pipeline`)

`Pipeline` chains multiple preprocessing steps such that the output of a transformer is automatically passed as the input of the next.

In [16]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler

### Chaining preprocessing steps

`fit_transform` fits each transformer in a sequence, returnin only the final transformed data.

In [17]:
X_pipe = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("minmax", MinMaxScaler()),
])

X_transformed = pipe.fit_transform(X_pipe)

X_transformed
    

array([[0. , 0. ],
       [0.5, 0.5],
       [1. , 1. ]])

### Applying fitted pipeline to new data

After fitting is completed, `transform` reuses parameters learned from original data, instead of repeat the process by fitting the preprocessing steps again.

In [18]:
new_data = np.array([
    [1.5, 15.0],
    [2.5, 25.0],
])

pipe.transform(new_data)

array([[0.25, 0.25],
       [0.75, 0.75]])

A pipeline could also end up with an estimator that provides `fit` and `predict`. As NumCompute does not implement a machine learning model, the example above only focuses on chaining preprocessing transformers.

# 9. Stats

'Stats' provides functions for basic statsical operations. Calling the apporiate function and taking in atleast a numpy array is all that is required. Axis is an optional variable, with a default to 0

In [19]:
from numcompute import mean, median, stan_dev, min, max

test_data = np.array([10, 5, 20, 15])

print(f"Data: {test_data}")
print(f"Mean: {mean(test_data)}")
print(f"Median: {median(test_data)}")
print(f"Standard Deviation: {stan_dev(test_data)}")
print(f"Minimum: {min(test_data)}")
print(f"Maximum: {max(test_data)}")

Data: [10  5 20 15]
Mean: 12.5
Median: 12.5
Standard Deviation: 5.5901699437494745
Minimum: 5
Maximum: 20


Histogram and quantile have bin and probability as options they also have defaults however these defaults are too large for this test data so they are included here

In [20]:
from numcompute import histogram,quantile

test_data = np.array([10, 5, 20, 15])
test_bins = 4
test_prob = 0.5

print(f"Data: {test_data} Bins: {test_bins} Probability:{test_prob}")
print(f"Histogram: {histogram(test_data,test_bins)}")
print(f"Quantile: {quantile(test_data,test_prob)}")

Data: [10  5 20 15] Bins: 4 Probability:0.5
Histogram: (array([1, 1, 1, 1]), array([ 5.  ,  8.75, 12.5 , 16.25, 20.  ]))
Quantile: 12.5


## 10. Metrics
These are implementations of basic machine learning metrics. Predicted results and actual results are required. Below is the binary classification examples and then regression examples

In [21]:
from numcompute.metrics import accuracy, precision, recall, f1score, confusion_matrix

pred_array = np.array([1,1,0,0])
true_array = np.array([1,0,0,0])

print(f"Predicted values: {pred_array} Actual Values: {true_array}")
print(f"Accuracy: {accuracy(pred_array,true_array)}")
print(f"Precision: {precision(pred_array, true_array)}")
print(f"Recall: {recall(pred_array, true_array)}")
print(f"f1score: {f1score(pred_array, true_array)}")
print(f"Confusion matrix: {confusion_matrix(pred_array, true_array)}")

Predicted values: [1 1 0 0] Actual Values: [1 0 0 0]
Accuracy: 0.75
Precision: 0.5
Recall: 1.0
f1score: 0.6666666666666666
Confusion matrix: (np.int64(1), np.int64(1), np.int64(2), np.int64(0))


In [ ]:
from numcompute.metrics import MSE

pred_array = np.array([2,1,0.5,0])
true_array = np.array([1,0.5,1,0])

print(f"Predicted values: {pred_array} Actual Values: {true_array}")
print(f"Mean Square Error: {MSE(pred_array, true_array)}")

## 11. Rank
The ranking tools in this library, compute average, ordinal and dense rankings for data as well as an additional percentile function to be able to identify values at given positions/percentiles. If a position lies between two points, interpolation methods such as linear, lower, higher and midpoint interpolations are used to find the percentile.


In [ ]:
from numcompute import rank, percentile

test_array = np.array([0, 2, 3, 2, 7, 2])

# Predicted Ordinal ranking
pred_arr_ord = np.array([1, 2, 5, 3, 6, 4])
# Predicted Average Ranking
pred_arr_avg = np.array([1, 3, 5, 3, 6, 3])
# Predicted Dense Ranking
pred_arr_dense = np.array([1, 2, 3, 2, 4, 2,])
# Predicted linear Percentile q = 30 (or 0.3)
pred_percentile = 0.2

print(f"Predicted values:\nAverage ranks: {pred_arr_avg}\nOrdinal ranks: {pred_arr_ord}\nDense ranks: {pred_arr_dense}")
print(f"\nPredicted Percentile, q = 30 (or 0.3), linear interpolation: {pred_percentile}")

print(f"\nComputed values:\nAverage ranks: {rank(test_array, "average")}\nOrdinal ranks: {rank(test_array, "ordinal")}\nDense ranks: {rank(test_array, "dense")}")
print(f"\nComputed percentile, q = 30 (or 0.3), linear interpolation: {percentile(test_array,0.3, "linear")}")


## 12. Benchmarking
The numcompute library also includes a Benchmarking tool called "benchit" which measures the average execution time of any function that is plugged into it. It also has a separate demo mode that enables compares various numcompute functions, which are implemented via vectorization, to a Python loop equivalent.

Note: With some functions a lot "0s" may be recorded despite the high precision used, this is because the operations are completed so quickly they are unable to be recorded properly by the timer used in the benchmarking process.

In most cases, performance is similar

In [ ]:
from numcompute import benchmarking
DEMO_FUNCS = ("stable_sort", "multi_key_sort", "median", "stan_dev", "imputer")

for func in DEMO_FUNCS:
    benchmarking.benchit(func,mode='demo', repeat=10)